In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

driver.get("http://localhost:5173/")
driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
driver.get("http://localhost:5173/login")

wait.until(EC.presence_of_element_located((By.ID, "username")))
driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
driver.find_element(By.ID, "password").send_keys("12345678")
driver.find_element(By.ID, "sign-in-btn").click()

time.sleep(3)
print("URL:", driver.current_url)
print("Page text:", driver.find_element(By.TAG_NAME, "body").text[:200])

In [ ]:
try:
    # Go to POS / Sales (sidebar button, verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'POS / Sales')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h2[text()='POS / Sales']")))

    # Sensitive medicines show a "Restricted" pill in their catalog row (verified in CatalogTable.jsx)
    wait.until(EC.presence_of_element_located((By.XPATH, "//tr[contains(@class, 'pos-row')]")))
    rows = [r for r in driver.find_elements(By.XPATH, "//tr[contains(@class, 'pos-row')]") if r.is_displayed() and "Restricted" in r.text]
    assert rows, "No sensitive (Restricted) medicine found in the catalog."
    row = rows[0]
    med_name = row.text.split("\n")[0]
    print("Sensitive medicine:", med_name)
    add = [b for b in row.find_elements(By.XPATH, ".//button[contains(., 'Add')]") if b.is_displayed() and b.is_enabled()]
    assert add, "Add button is disabled for the sensitive medicine."
    add[0].click()
    time.sleep(2)

    # Trigger the warning by completing the sale
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Complete Sale')]"))).click()
    time.sleep(3)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//*[text()='Sensitive Medicine Approval']")))
    print("Approval UI shown: Sensitive Medicine Approval")

    # Perform the approval through the real UI button (verified in InteractionReviewModal.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Reviewed')]"))).click()
    time.sleep(3)

    # The sale must now proceed: receipt modal with "Sale Completed" (verified in ReceiptModal.jsx)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//*[text()='Sale Completed']")))
    invoice = driver.find_element(By.XPATH, "//*[contains(text(), 'Invoice #')]").text
    print("Sale proceeded after approval:", invoice)
    print("Current URL:", driver.current_url)
    print("PASS: Sensitive Medicine Approval")
except Exception as e:
    print("FAIL: Sensitive Medicine Approval")
    print("Error:", e)
    driver.save_screenshot("22_sensitive_approval_FAIL.png")

In [ ]:
driver.quit()